# 05 — Exploratory spatial LR label association

Scores are mean log-expression products on directed, same-biopsy edges for each cluster pair. Cluster labels are shuffled within each biopsy, keeping geometry and expression fixed. BH correction covers the full family of sample × cluster-pair × available LR-pair tests. This is neither proof of communication nor a patient-level rejection test; expression-derived clusters also make the analysis exploratory.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from src import preprocessing, utils, visualization
from src.spatial import build_spatial_graph, validate_graph, spatial_lr
settings = utils.load_settings()
params = settings['params']
seed = params['seed']
sc.settings.set_figure_params(dpi=params['visualization']['figure_dpi'], facecolor='white')

In [ ]:
run = utils.begin_stage(settings, 'adata_final.h5ad', 'adata_profiles.h5ad')
adata = sc.read_h5ad(run / 'processed' / 'adata_profiles.h5ad')
pair_table = pd.read_csv(ROOT / 'config' / 'ligand_receptor_pairs.csv')
pair_table['available'] = pair_table['ligand'].isin(adata.var_names) & pair_table['receptor'].isin(adata.var_names)
pair_table.to_csv(run / 'tables' / 'lr_pair_availability.csv', index=False)
lr_cfg = params['ligand_receptor']
results = spatial_lr(adata, pair_table[['ligand', 'receptor']].itertuples(index=False, name=None), n_perms=lr_cfg['n_perms'], seed=seed)
results.to_csv(run / 'tables' / 'spatial_lr_all.csv', index=False)
significant = results[(results['padj'] < lr_cfg['fdr']) & (results['score'] > 0) & (results['n_edges'] > 0)].copy()
significant.to_csv(run / 'tables' / 'spatial_lr_fdr.csv', index=False)
print(f'{pair_table.available.sum()} LR pairs; {len(results)} hypotheses; {len(significant)} FDR-filtered associations')
visualization.plot_ligand_receptor(significant, top_n=20, save_path=run / 'figures' / 'spatial_lr_fdr.png')
plt.show()

In [ ]:
# Each row is a biopsy-level descriptive aggregate; no disease p-value or pooled logFC.
weighted = results.assign(edge_product_sum=results['score'] * results['n_edges'])
keys = ['sample', 'condition', 'rejection_type', 'ligand', 'receptor']
summary = weighted.groupby(keys, observed=True)[['edge_product_sum', 'n_edges']].sum().reset_index()
summary['mean_edge_product'] = np.divide(summary['edge_product_sum'], summary['n_edges'], out=np.full(len(summary), np.nan), where=summary['n_edges'] > 0)
summary.to_csv(run / 'tables' / 'spatial_lr_descriptive_by_sample.csv', index=False)
adata.uns['lr_analysis'] = {'method': 'within_sample_edge_product_label_permutation', 'expression_layer': 'lognorm', 'n_perms': lr_cfg['n_perms'], 'seed': seed, 'fdr_method': 'BH_global_across_samples_cluster_pairs_lr_pairs', 'n_hypotheses': len(results), 'n_fdr_filtered': len(significant), 'results_file': 'spatial_lr_all.csv'}
utils.save_checkpoint(adata, run / 'processed' / 'adata_final.h5ad')